# 2.1 DataFrame изнутри: код страницы и упражнения

Аналитика 101 · блок «Дата аналитика» · тема 02 «Pandas и подготовка данных» · страница 2.1.

В ноутбуке шесть блоков кода со страницы и пять упражнений на `loc`, `iloc`, маски и типы. Данные учебные: первая ячейка кода сама собирает выгрузку из 100 000 заказов с зерном 9, скачивать ничего не нужно. Та же выгрузка лежит рядом файлом `orders_2026q1.csv`.

**Как работать.** Выполняйте ячейки по порядку сверху вниз. В упражнении замените `...` своим кодом и запустите ячейку проверки под ним. Она напечатает «верно» или покажет, что не сошлось. Решения с разбором — в `dataframe_inside_solutions.ipynb`.

Вывод под блоками кода получен на Python 3.13 и pandas 3.0. Время в замерах у вас будет другим. В pandas 2 другими будут и типы строк, и поведение цепочек присваивания — об этом на странице.

In [1]:
import platform

import numpy as np
import pandas as pd

print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

Python 3.13.18 | numpy 2.5.3 | pandas 3.0.5


## Код страницы

### Учебная выгрузка: info() и memory_usage()

In [2]:
import numpy as np
import pandas as pd

# Учебная выгрузка заказов: 100 000 строк, январь–март 2026
rng = np.random.default_rng(9)            # зерно: при каждом запуске те же данные
n = 100_000
orders = pd.DataFrame({
    "order_id": np.arange(10001, 10001 + n),
    "date": pd.Timestamp("2026-01-01")
            + pd.to_timedelta(rng.integers(0, 90, n), unit="D"),
    "brand": rng.choice(["Сова", "Кедр", "Луч", "Бриз"], n),
    "country": rng.choice(["BY", "RU", "KZ", "UZ"], n),
    "channel": rng.choice(["органика", "реклама", "рассылка"], n),
    "amount": rng.integers(5, 400, n) * 10,             # от 50 до 3 990 ₽
    "status": rng.choice(["paid", "cancelled"], n, p=[0.85, 0.15]),
})

orders.info()                                           # типы, пропуски, память
print()
kb = orders.memory_usage(deep=True) / 1024              # байты → КБ
print(kb.round(1))

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   order_id  100000 non-null  int64         
 1   date      100000 non-null  datetime64[us]
 2   brand     100000 non-null  str           
 3   country   100000 non-null  str           
 4   channel   100000 non-null  str           
 5   amount    100000 non-null  int64         
 6   status    100000 non-null  str           
dtypes: datetime64[us](1), int64(2), str(4)
memory usage: 8.2 MB

Index          0.1
order_id     781.2
date         781.2
brand       1513.8
country      976.6
channel     2278.5
amount       781.2
status      1245.5
dtype: float64


Индекс по умолчанию — RangeIndex: он хранит только начало, конец и шаг, поэтому занимает 0,1 КБ. Числа и даты — ровно по 8 байт на значение, строки str — байты текста в UTF-8 плюс смещения.

### Перевод типов: str, object, category и числа из строк

In [3]:
from timeit import timeit

kb = lambda x: round(np.sum(x.memory_usage(deep=True)) / 1024)   # КБ

# Один и тот же столбец brand в трёх типах
variants = {
    "str": orders["brand"],                       # строки по умолчанию в pandas 3
    "object": orders["brand"].astype(object),     # так строки хранил pandas 2
    "category": orders["brand"].astype("category"),   # 4 бренда и коды 0–3
}
for name, s in variants.items():
    ms = timeit(lambda: s == "Сова", number=20) / 20 * 1000
    print(f"{name:9}{kb(s):6} КБ   s == 'Сова': {ms:.2f} мс")

# Все повторяющиеся строковые столбцы — в category
typed = orders.astype({c: "category" for c in ["brand", "country", "channel", "status"]})
print("вся таблица:", kb(orders), "КБ →", kb(typed), "КБ")
print(typed["brand"].cat.categories.tolist(), typed["brand"].cat.codes.dtype)

# Числа, пришедшие строками: одно «н/д» делает весь столбец текстом
raw = pd.Series(["3430", "400", "н/д", "1 300"])
print(raw.dtype, raw.sum())                       # sum() склеивает строки
fixed = pd.to_numeric(raw.str.replace(" ", ""), errors="coerce")
print(fixed.tolist(), fixed.dtype)
print(pd.Series([3430, 400, "н/д"]).dtype)         # числа вперемешку со строкой

str        1514 КБ   s == 'Сова': 1.56 мс


object     7178 КБ   s == 'Сова': 8.03 мс
category     98 КБ   s == 'Сова': 0.16 мс


вся таблица: 8358 КБ → 2735 КБ
['Бриз', 'Кедр', 'Луч', 'Сова'] int8
str 3430400н/д1 300
[3430.0, 400.0, nan, 1300.0] float64
object


Время зависит от компьютера, а соотношения повторяются: object медленнее str в несколько раз, category быстрее на порядок. Перевод в число дал float64, потому что NaN — значение с плавающей точкой.

### Одна выборка четырьмя способами

In [4]:
# Оплаченные заказы из Беларуси дороже 3 000 ₽: номер и сумма
cols = ["order_id", "amount"]
mask = ((orders["status"] == "paid") & (orders["country"] == "BY")
        & (orders["amount"] > 3000))

a = orders.loc[mask, cols]                                # 1. loc: маска и имена
b = orders[mask][cols]                                    # 2. две пары скобок
c = orders.query("status == 'paid' and country == 'BY' "
                 "and amount > 3000")[cols]               # 3. query: условие строкой
rows = np.flatnonzero(mask.to_numpy())                    # позиции, где маска True
d = orders.iloc[rows, [orders.columns.get_loc(x) for x in cols]]   # 4. iloc

print(len(a), a.equals(b), a.equals(c), a.equals(d))
print(a.head(3))
print(rows[:3])

5292 True True True
     order_id  amount
70      10071    3870
103     10104    3180
110     10111    3450
[ 70 103 110]


Индекс здесь RangeIndex, поэтому позиции 70, 103, 110 совпадают с метками. После сортировки способ 4 пришлось бы пересчитывать, а способы 1–3 работают с метками и условием и не зависят от порядка строк.

### Замер: цикл по строкам против векторного выражения

In [5]:
import time

def best_ms(f, repeat=3):
    """Лучшее время из нескольких запусков, мс."""
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        result = f()
        times.append(time.perf_counter() - t0)
    return result, min(times) * 1000

def by_iterrows():                        # строка за строкой, каждая — новая Series
    total = 0
    for _, row in orders.iterrows():
        if row["status"] == "paid":
            total += row["amount"]
    return total

def by_zip():                             # цикл Python по двум столбцам
    total = 0
    for status, amount in zip(orders["status"], orders["amount"]):
        if status == "paid":
            total += amount
    return total

def vectorized():                         # маска и сумма над целыми столбцами
    return orders.loc[orders["status"] == "paid", "amount"].sum()

print(f"{'способ':10}{'выручка, ₽':>14}{'мс':>10}{'быстрее iterrows':>19}")
base = None
for name, f in [("iterrows", by_iterrows), ("zip", by_zip), ("векторно", vectorized)]:
    total, ms = best_ms(f)
    base = base or ms
    print(f"{name:10}{total:>14,}{ms:>10.1f}{base / ms:>18.0f}×".replace(",", " "))

способ        выручка, ₽        мс   быстрее iterrows


iterrows     171 247 920    2592.7                 1×


zip          171 247 920      81.7                32×
векторно     171 247 920       2.3              1121×


Ответ у всех трёх способов один, различается только время. Абсолютные миллисекунды зависят от компьютера, порядок разницы — нет.

### Часовой пояс сдвигает дату заказа

In [6]:
# Время заказов записано в UTC, а сутки отчёта — по местному времени
utc = pd.Series(pd.to_datetime(["2026-03-28 22:30", "2026-03-29 08:10"]))
print(utc.dtype)                                    # наивное время: пояс неизвестен
utc = utc.dt.tz_localize("UTC")                     # объявить: это UTC
print(utc.dtype)
for tz in ["UTC", "Europe/Minsk", "Asia/Tashkent"]:
    local = utc.dt.tz_convert(tz)                   # тот же момент, другие часы
    print(f"{tz:14}", local.dt.strftime("%d.%m %H:%M").tolist())
try:
    utc > pd.Timestamp("2026-03-29")                # сравнить с датой без пояса
except TypeError as e:
    print("TypeError:", e)

datetime64[us]
datetime64[us, UTC]
UTC            ['28.03 22:30', '29.03 08:10']
Europe/Minsk   ['29.03 01:30', '29.03 11:10']
Asia/Tashkent  ['29.03 03:30', '29.03 13:10']
TypeError: Invalid comparison between dtype=datetime64[us, UTC] and Timestamp


Первый заказ в UTC сделан 28 марта, а по минскому и ташкентскому времени — 29-го. Дневная выручка зависит от того, в каком поясе считать сутки.

Последний блок страницы меняет способ показа предупреждений до конца ноутбука: дальше каждое предупреждение будет печататься одной строкой. На странице под этим блоком есть ещё вывод того же кода в pandas 2.3.3.

### Цепочка присваивания в pandas 3 и pandas 2

In [7]:
import warnings
warnings.simplefilter("always")                     # показывать каждое предупреждение
warnings.showwarning = lambda m, c, *a, **k: print(f"   {c.__name__}: {str(m).strip().splitlines()[0]}")

df = orders.head(8).sort_values("amount", ascending=False)
cancelled = df["status"] == "cancelled"             # одна строка: метка 4, 1 300 ₽

df[cancelled]["amount"] = 0                         # A: строки, потом столбец
print("A:", df.loc[4, "amount"])
df["amount"][cancelled] = 0                         # B: столбец, потом строки
print("B:", df.loc[4, "amount"])
paid = df[df["status"] == "paid"]                   # C: новая таблица из выборки
paid["amount"] = paid["amount"] * 2
print("C:", df.loc[0, "amount"], paid.loc[0, "amount"])
df.loc[cancelled, "amount"] = 0                     # D: один loc — строки и столбец
print("D:", df.loc[4, "amount"])

   ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
A: 1300
   ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
B: 1300
C: 3430 6860
D: 0


Третья строка кода печатает только первую строку каждого предупреждения, чтобы вывод поместился. В pandas 2 строка B изменила таблицу, а A — нет; в pandas 3 не работает ни одна цепочка, работает только D.

## Упражнения

Таблица для упражнений — та же, что в анимации и интерактиве: первые восемь заказов выгрузки, отсортированные по сумме. Собираем её заново: блок про цепочки присваивания обнулил в `df` одну сумму.

In [8]:
df = orders.head(8).sort_values("amount", ascending=False)
print(df[["date", "brand", "amount", "status"]])

        date brand  amount     status
0 2026-02-07  Кедр    3430       paid
6 2026-03-02   Луч    3000       paid
5 2026-02-24  Кедр    2640       paid
3 2026-01-26  Бриз    2490       paid
4 2026-01-11  Сова    1300  cancelled
7 2026-03-11  Кедр    1240       paid
2 2026-03-28  Кедр     910       paid
1 2026-03-20   Луч     400       paid


### Упражнение 1. Позиция или метка

Запишите в `second` сумму второго по величине заказа, а в `label2` — сумму заказа с меткой 2.

In [ ]:
second = ...   # ваш код: вторая строка сверху
label2 = ...   # ваш код: строка с меткой 2

In [ ]:
assert second == 3000, f"Второй по величине заказ — 3 000 ₽, получено {second!r}. Какая по счёту — это iloc"
assert label2 == 910, f"У заказа с меткой 2 сумма 910 ₽, получено {label2!r}. Какая по имени — это loc"
print("Упражнение 1: верно")

### Упражнение 2. Срез по позициям и столбцы по именам

Соберите таблицу `top3`: три самых дорогих заказа, только столбцы `brand` и `amount`.

In [ ]:
top3 = ...   # ваш код: первые три строки и два столбца

In [ ]:
assert isinstance(top3, pd.DataFrame), "top3 должна быть таблицей (DataFrame)"
assert list(top3.columns) == ["brand", "amount"], f"Нужны столбцы brand и amount, получено {list(top3.columns)}"
assert list(top3.index) == [0, 6, 5], f"Три самых дорогих заказа носят метки 0, 6 и 5, получено {list(top3.index)}"
print("Упражнение 2: верно")

### Упражнение 3. Маска из двух условий

Соберите таблицу `kedr`: оплаченные заказы бренда «Кедр» дороже 1 000 ₽.

In [ ]:
kedr = ...   # ваш код: три условия, соединённые через &

In [ ]:
assert isinstance(kedr, pd.DataFrame), "kedr должна быть таблицей (DataFrame)"
assert list(kedr.index) == [0, 5, 7], f"Подходят заказы с метками 0, 5 и 7, получено {list(kedr.index)}"
print("Упражнение 3: верно")

### Упражнение 4. Присваивание, которое доходит до таблицы

В таблице `df` один отменённый заказ. Поставьте ему сумму 0 так, чтобы изменилась сама `df`.

In [ ]:
...   # ваш код: одна строка с loc

In [ ]:
assert df.loc[4, "amount"] == 0, f"У отменённого заказа (метка 4) сумма должна стать 0, сейчас {df.loc[4, 'amount']}"
assert df["amount"].sum() == 14110, "Остальные суммы меняться не должны: всего должно остаться 14 110 ₽"
print("Упражнение 4: верно")

### Упражнение 5. Числа, которые пришли строками

Столбец `raw` прочитан из файла текстом: в одной сумме пробел между разрядами, вместо другой стоит «н/д». Получите числовой столбец `fixed` типа `float64`, где «н/д» стало пропуском, и сумму известных значений `total`.

In [ ]:
raw = pd.Series(["3 430", "400", "н/д", "1 300"])
print(raw.dtype, raw.sum())

In [ ]:
fixed = ...   # ваш код: убрать пробелы и перевести в числа
total = ...   # ваш код: сумма известных значений

In [ ]:
assert isinstance(fixed, pd.Series) and fixed.dtype == "float64", "fixed должен быть столбцом типа float64"
assert fixed.isna().sum() == 1, "На месте «н/д» должен стоять один пропуск NaN"
assert total == 5130, f"Сумма известных значений — 5 130, получено {total!r}"
print("Упражнение 5: верно")